In [ ]:
import os
import wfdb
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.model_selection import train_test_split

from src.preprocessing import bandpass_filter
from src.segmentation import detect_r_peaks, get_sliding_windows, resample_signal, normalize_per_window

In [ ]:
DATA_PATH = '../data/mit-bih-arrhythmia-database-1.0.0/'
SAVE_PATH = '/content/drive/MyDrive/File Tugas Kuliah/TA 3/'
os.makedirs(SAVE_PATH, exist_ok=True)

NUM_PEAKS = 3
TARGET_LEN = 600
FS = 360

In [ ]:
record_no = '100'
record = wfdb.rdrecord(os.path.join(DATA_PATH, record_no))
annotation = wfdb.rdann(os.path.join(DATA_PATH, record_no), 'atr')

signal = record.p_signal[:, 0]
waktu = np.arange(len(signal)) / FS

filtered_signal = bandpass_filter(signal, lowcut=0.5, highcut=45.0, fs=FS)

plt.figure(figsize=(12, 5))
plt.plot(waktu[:1800], signal[:1800], label="Sinyal Asli (Mentah)", alpha=0.6)
plt.plot(waktu[:1800], filtered_signal[:1800], label="Sinyal Filtered", linewidth=1.5)
plt.title(f"Visualisasi Sinyal ECG - Record {record_no}")
plt.xlabel("Waktu (Detik)")
plt.ylabel("Amplitudo (mV)")
plt.legend()
plt.grid()
plt.show()

In [ ]:
records = ['100', '101', '102', '103', '104', '105', '106', '107', '108', '109',
           '111', '112', '113', '114', '115', '116', '118', '119', '121', '122',
           '123', '124', '200', '201', '202', '203', '205', '207', '208', '209',
           '210', '212', '213', '214', '215', '217', '219', '220', '221', '222',
           '223', '228', '230', '231', '232', '233', '234']

X_list = []
y_list = []

print(f"Memulai ekstraksi {NUM_PEAKS}R windows...")

for rec in records:
    try:
        sig, fields = wfdb.rdsamp(os.path.join(DATA_PATH, rec))
        ann = wfdb.rdann(os.path.join(DATA_PATH, rec), 'atr')
    except:
        continue

    signal_mlii = sig[:, 0]
    filtered_sig = bandpass_filter(signal_mlii, 0.5, 45.0, FS)
    r_peaks = detect_r_peaks(filtered_sig, FS, filtered=True)
    windows, _ = get_sliding_windows(filtered_sig, r_peaks, FS, num_peaks=NUM_PEAKS)

    for start, end, win_sig in windows:
        win_resampled = resample_signal(win_sig, TARGET_LEN)

        labels_in_window = [sym for pos, sym in zip(ann.sample, ann.symbol) if start <= pos <= end]
        if labels_in_window:

            most_common_label = Counter(labels_in_window).most_common(1)[0][0]
            X_list.append(win_resampled)
            y_list.append(most_common_label)

X = np.array(X_list)
y = np.array(y_list)

X = normalize_per_window(X)
X = np.expand_dims(X, axis=-1)

print(f"Total data berhasil diekstrak: {X.shape}")
print(f"Distribusi kelas: {Counter(y)}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Shape X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"Shape X_test: {X_test.shape}, y_test: {y_test.shape}")

with open(f'{SAVE_PATH}X_train_{NUM_PEAKS}R.pkl', 'wb') as f:
    pickle.dump(X_train, f)
with open(f'{SAVE_PATH}y_train_{NUM_PEAKS}R.pkl', 'wb') as f:
    pickle.dump(y_train, f)
with open(f'{SAVE_PATH}X_test_{NUM_PEAKS}R.pkl', 'wb') as f:
    pickle.dump(X_test, f)
with open(f'{SAVE_PATH}y_test_{NUM_PEAKS}R.pkl', 'wb') as f:
    pickle.dump(y_test, f)

print(f"\nData {NUM_PEAKS}R telah disimpan ke {SAVE_PATH} dan siap digunakan untuk eksperimen model.")